# rebuilding embeddings: bge-m3 а не minilm 

bge-m3 wins on both EN and RU so switching to it

| model | EN hit@5 | RU hit@5 |
|-------|----------|----------|
| all-MiniLM-L6-v2 | 32% | 2% |
| bge-m3 | 38% | 37% |

saves baseline metrics, filters nsfw (228 memes), builds 4 indices on bge-m3, compares old vs new


In [11]:
import os
os.chdir(os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else "."))
import json
import numpy as np
import time
import torch
from pathlib import Path

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", device)
print("CWD:", os.getcwd())


Device: mps
CWD: /Users/amirhamidullin/PycharmProjects/coursework3


## 1. Сохранение baseline (MiniLM)

In [ ]:
BASELINE = {
    "model": "all-MiniLM-L6-v2",
    "EN_hit1": 0.17, "EN_hit5": 0.32, "EN_hit10": 0.37, "EN_mrr10": 0.2361,
    "RU_hit1": 0.01, "RU_hit5": 0.02, "RU_hit10": 0.02, "RU_mrr10": 0.0150,
    "note": "single caption index, baseline"
}

from shutil import copy2
backup_dir = Path("data/processed/backup_minilm")
backup_dir.mkdir(exist_ok=True)

for f in ["emb_ocr.npy", "emb_caption.npy", "emb_keywords.npy", "emb_image.npy",
          "faiss_ocr.index", "faiss_caption.index", "faiss_keywords.index", "faiss_image.index"]:
    src = Path("data/processed") / f
    if src.exists():
        copy2(src, backup_dir / f)
        print(f"  backed up {f}")

with open("eval/baseline_metrics.json", "w") as f:
    json.dump(BASELINE, f, indent=2)

print(f"\nbaseline saved: en hit@5={BASELINE['EN_hit5']:.0%}, ru hit@5={BASELINE['RU_hit5']:.0%}")


  Backup: emb_ocr.npy
  Backup: emb_caption.npy
  Backup: emb_keywords.npy
  Backup: emb_image.npy
  Backup: faiss_ocr.index
  Backup: faiss_caption.index
  Backup: faiss_keywords.index
  Backup: faiss_image.index

Baseline сохранён в eval/baseline_metrics.json
  EN Hit@5: 32%
  RU Hit@5: 2%


## 2. Загрузка данных + NSFW фильтрация

In [ ]:
with open("data/processed/vqa_annotations_v2.jsonl") as f:
    all_records = [json.loads(l) for l in f]

records = [r for r in all_records if not r.get("is_nsfw", False)]
nsfw_count = len(all_records) - len(records)

print(f"total: {len(all_records)}, nsfw: {nsfw_count}, clean: {len(records)}")

old_to_new = {}
new_idx = 0
for old_idx, r in enumerate(all_records):
    if not r.get("is_nsfw", False):
        old_to_new[old_idx] = new_idx
        new_idx += 1

print(f"index mapping: {len(old_to_new)} entries")


Всего: 9998
NSFW: 228
Чистых: 9770
Маппинг: 9770 записей


## 3. Подготовка текстов

In [ ]:
def extract_ocr(records):
    texts = []
    for r in records:
        ocr = r.get("ocr_normalized", "").strip() or r.get("ocr_text", "").strip()
        texts.append(ocr if len(ocr) > 2 else "")
    print(f"ocr: {sum(1 for t in texts if t)}/{len(texts)} non-empty")
    return texts

def extract_caption(records):
    texts = []
    for r in records:
        parts = []
        cap = r.get("caption", "")
        if cap:
            parts.append(cap)
        idea = r.get("main_idea", "")
        if idea and "one sentence" not in idea.lower():
            parts.append(idea)
        text = " ".join(parts).strip()
        if not text:
            text = r.get("raw_response", r.get("filename", ""))[:200]
        texts.append(text)
    print(f"caption: {sum(1 for t in texts if t)}/{len(texts)} non-empty")
    return texts

def extract_keywords(records):
    texts = []
    for r in records:
        parts = []
        objects = r.get("objects", [])
        if isinstance(objects, list) and objects:
            clean = [str(o) for o in objects if str(o) not in ("key", "objects", "max 5")]
            if clean:
                parts.append(", ".join(clean[:8]))
        objects_det = r.get("objects_detailed", [])
        if isinstance(objects_det, list) and objects_det:
            existing = set(str(o).lower() for o in objects)
            new_objs = [str(o) for o in objects_det if str(o).lower() not in existing][:5]
            if new_objs:
                parts.append(", ".join(new_objs))
        tone = r.get("tone", "")
        if tone:
            parts.append(tone.split("/")[0].strip() if "/" in tone and len(tone) > 20 else tone)
        texts.append(". ".join(parts) if parts else "")
    print(f"keywords: {sum(1 for t in texts if t)}/{len(texts)} non-empty")
    return texts

ocr_texts = extract_ocr(records)
caption_texts = extract_caption(records)
keyword_texts = extract_keywords(records)

for i in range(3):
    print(f"\n{records[i].get('filename', '')}")
    print(f"ocr: {ocr_texts[i][:80]}")
    print(f"caption: {caption_texts[i][:80]}")
    print(f"keywords: {keyword_texts[i][:80]}")


OCR: 4741/9770 непустых
Caption: 9770/9770 непустых
Keywords: 9747/9770 непустых

--- e9d93d499127.jpg ---
  OCR:      I WISH I WAS YOUR MATH HOMEWORK BECAUSE THEN I'D BE HARD AND YOU'D BE DOING ME O
  Caption:  I wish I was your math homework because then I'd be hard and you'd be doing me o
  Keywords: person wearing red shirt, person wearing black shirt, person with brown hair, pe

--- e0be03878bc0.png ---
  OCR:      САТАНИСТОКА
  Caption:  A person with goat horns and a black robe is shown in a stylized poster with the
  Keywords: person with goat horns, black robe, goat horns, text "САТАНИСТО4КА". Person with

--- 81ee114dfc33.jpg ---
  OCR:      Venture Bros is the best show ever. You can't change my mind.
  Caption:  An animated character is sitting at a table with a sign that says 'Venture Bros 
  Keywords: table, sign, character. animated character, microphone, camera, text on sign, br


## 4. Загрузка bge-m3 + генерация эмбеддингов

In [ ]:
from sentence_transformers import SentenceTransformer


t0 = time.time()
model = SentenceTransformer("BAAI/bge-m3", device=device)


def encode_texts(texts, name):
    empty_mask = [t.strip() == "" for t in texts]
    safe_texts = [t if t.strip() else "empty" for t in texts]
    t0 = time.time()
    emb = model.encode(safe_texts, show_progress_bar=True, batch_size=64,
                       normalize_embeddings=True, device=device)
    for i, is_empty in enumerate(empty_mask):
        if is_empty:
            emb[i] = np.zeros(emb.shape[1])
    print(f"  {name}: {emb.shape}, {time.time()-t0:.1f}с")
    return emb.astype(np.float32)

emb_ocr = encode_texts(ocr_texts, "OCR")
emb_caption = encode_texts(caption_texts, "Caption")
emb_keywords = encode_texts(keyword_texts, "Keywords")


Загрузка bge-m3...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Загружена за 10.7с
Encode OCR: 9770 текстов...


Batches:   0%|          | 0/153 [00:00<?, ?it/s]

  OCR: (9770, 1024), 25.2с
Encode Caption: 9770 текстов...


Batches:   0%|          | 0/153 [00:00<?, ?it/s]

  Caption: (9770, 1024), 64.5с
Encode Keywords: 9770 текстов...


Batches:   0%|          | 0/153 [00:00<?, ?it/s]

  Keywords: (9770, 1024), 72.2с


## 5. image эмбеддинги 

In [ ]:
old_img = np.load("data/processed/backup_minilm/emb_image.npy")

if old_img.shape[0] == len(all_records):
    clean_indices = [i for i, r in enumerate(all_records) if not r.get("is_nsfw", False)]
    emb_image = old_img[clean_indices]
else:
    from PIL import Image as PILImage
    from transformers import CLIPProcessor, CLIPModel
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    clip_model.eval()
    embeddings = []
    for r in records:
        img_path = Path(r.get("source_path", ""))
        try:
            img = PILImage.open(img_path).convert("RGB")
            inputs = clip_proc(images=img, return_tensors="pt")
            with torch.no_grad():
                out = clip_model.vision_model(pixel_values=inputs["pixel_values"])
                feat = clip_model.visual_projection(out.pooler_output)
                feat = feat / feat.norm(dim=-1, keepdim=True)
            embeddings.append(feat.numpy().flatten().astype(np.float32))
        except:
            embeddings.append(np.zeros(512, dtype=np.float32))
    emb_image = np.stack(embeddings)
    print(f"clip recompute  {emb_image.shape}")


Старые image embeddings: (9770, 512)
Размер не совпадает (9770 != 9998), пересчёт CLIP...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

/Users/amirhamidullin/PycharmProjects/recsys/DeepRecSys/.venv/lib/python3.13/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


CLIP пересчитан: (9770, 512)


## 6. Сохранение эмбеддингов + FAISS индексы

In [ ]:
import faiss

OUT = Path("data/processed")

np.save(OUT / "emb_ocr.npy", emb_ocr)
np.save(OUT / "emb_caption.npy", emb_caption)
np.save(OUT / "emb_keywords.npy", emb_keywords)
np.save(OUT / "emb_image.npy", emb_image)

for name, emb in [("ocr", emb_ocr), ("caption", emb_caption),
                   ("keywords", emb_keywords), ("image", emb_image)]:
    idx = faiss.IndexFlatIP(emb.shape[1])
    idx.add(emb)
    faiss.write_index(idx, str(OUT / f"faiss_{name}.index"))
    print(f"faiss_{name}.index: {idx.ntotal} vectors, dim={emb.shape[1]}")

with open(OUT / "index_metadata.jsonl", "w", encoding="utf-8") as f:
    for r in records:
        meta = {
            "filename": r.get("filename", ""),
            "source_path": r.get("source_path", ""),
            "caption": r.get("caption", ""),
            "ocr_text": r.get("ocr_text", "")[:200],
            "objects": r.get("objects", []),
            "tone": r.get("tone", ""),
            "main_idea": r.get("main_idea", ""),
            "source_type": r.get("source_type", ""),
        }
        f.write(json.dumps(meta, ensure_ascii=False) + "\n")

print(f"\nmetadata saved: {len(records)} records")


faiss_ocr.index: 9770 vectors, dim=1024
faiss_caption.index: 9770 vectors, dim=1024
faiss_keywords.index: 9770 vectors, dim=1024
faiss_image.index: 9770 vectors, dim=512

Метаданные: 9770 записей
Все файлы сохранены!


## 7. Быстрая проверка: bge-m3 vs MiniLM baseline

In [ ]:
with open("eval/language_experiment_queries.json") as f:
    queries = json.load(f)

valid_queries = []
for q in queries:
    new_idx = old_to_new.get(q["index"])
    if new_idx is not None:
        q_copy = dict(q)
        q_copy["index"] = new_idx
        valid_queries.append(q_copy)

print(f"queries after nsfw filter: {len(valid_queries)}/{len(queries)}")

en_texts = [q.get("query_en", "").strip() or q["caption"][:100] for q in valid_queries]
ru_texts = [q.get("query_ru", "").strip() or q["caption"][:100] for q in valid_queries]

en_emb = model.encode(en_texts, normalize_embeddings=True, device=device)
ru_emb = model.encode(ru_texts, normalize_embeddings=True, device=device)

def calc_hits(q_emb, db_emb, queries_list):
    sims = q_emb @ db_emb.T
    hits = {1: 0, 5: 0, 10: 0}
    mrr = 0
    for i, q in enumerate(queries_list):
        ranking = np.argsort(-sims[i])
        pos = np.where(ranking == q["index"])[0]
        if len(pos):
            pos = pos[0] + 1
            for k in hits:
                if pos <= k: hits[k] += 1
            if pos <= 10: mrr += 1.0 / pos
    n = len(queries_list)
    return {k: v/n for k, v in hits.items()}, mrr/n

en_hits, en_mrr = calc_hits(en_emb, emb_caption, valid_queries)
ru_hits, ru_mrr = calc_hits(ru_emb, emb_caption, valid_queries)

print(f"\nbge-m3 caption index:")
print(f"  EN: hit@1={en_hits[1]:.0%} hit@5={en_hits[5]:.0%} hit@10={en_hits[10]:.0%} mrr={en_mrr:.4f}")
print(f"  RU: hit@1={ru_hits[1]:.0%} hit@5={ru_hits[5]:.0%} hit@10={ru_hits[10]:.0%} mrr={ru_mrr:.4f}")

print(f"\nbaseline (MiniLM):")
print(f"  EN: hit@5={BASELINE['EN_hit5']:.0%}  mrr={BASELINE['EN_mrr10']:.4f}")
print(f"  RU: hit@5={BASELINE['RU_hit5']:.0%}  mrr={BASELINE['RU_mrr10']:.4f}")

print(f"\ngain EN hit@5: +{en_hits[5] - BASELINE['EN_hit5']:.0%}")
print(f"gain RU hit@5: +{ru_hits[5] - BASELINE['RU_hit5']:.0%}")


Запросов после фильтрации: 100/100

Результаты: bge-m3 (caption index, чистые данные)
  EN: Hit@1=24% Hit@5=39% Hit@10=44% MRR=0.3018
  RU: Hit@1=13% Hit@5=37% Hit@10=43% MRR=0.2296

Baseline (MiniLM):
  EN: Hit@5=32%  MRR=0.2361
  RU: Hit@5=2%  MRR=0.0150

Прирост EN Hit@5: +7%
Прирост RU Hit@5: +35%


## 8. Сравнение: deepvk/USER-bge-m3 vs bge-m3

deepvk/USER-bge-m3 — дообучка bge-m3 на русских данных VKontakte.
В caption-эксперименте: EN=40%, RU=37% (bge-m3: EN=38%, RU=37%).

Проверим на multi-vecto  может разница проявится на OCR/keywords.


In [ ]:
from sentence_transformers import SentenceTransformer
import time

t0 = time.time()
model_dv = SentenceTransformer("deepvk/USER-bge-m3", device=device)
print(f"deepvk/USER-bge-m3 loaded in {time.time()-t0:.1f}s")

def encode_dv(texts, name):
    empty_mask = [t.strip() == "" for t in texts]
    safe_texts = [t if t.strip() else "empty" for t in texts]
    t0 = time.time()
    emb = model_dv.encode(safe_texts, show_progress_bar=True, batch_size=64,
                          normalize_embeddings=True, device=device)
    for i, is_empty in enumerate(empty_mask):
        if is_empty:
            emb[i] = np.zeros(emb.shape[1])
    print(f"  {name}: {emb.shape}, {time.time()-t0:.1f}s")
    return emb.astype(np.float32)

emb_ocr_dv = encode_dv(ocr_texts, "ocr")
emb_caption_dv = encode_dv(caption_texts, "caption")
emb_keywords_dv = encode_dv(keyword_texts, "keywords")


Загрузка deepvk/USER-bge-m3...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Загружена за 8.6с
Encode OCR-deepvk: 9770 текстов...


Batches:   0%|          | 0/153 [00:00<?, ?it/s]

  OCR-deepvk: (9770, 1024), 29.0с
Encode Caption-deepvk: 9770 текстов...


Batches:   0%|          | 0/153 [00:00<?, ?it/s]

  Caption-deepvk: (9770, 1024), 67.4с
Encode Keywords-deepvk: 9770 текстов...


Batches:   0%|          | 0/153 [00:00<?, ?it/s]

  Keywords-deepvk: (9770, 1024), 77.5с


In [ ]:

en_emb_dv = model_dv.encode(en_texts, normalize_embeddings=True, device=device)
ru_emb_dv = model_dv.encode(ru_texts, normalize_embeddings=True, device=device)

print(f"{'Индекс':<15} {'bge-m3 EN':>10} {'bge-m3 RU':>10} {'deepvk EN':>10} {'deepvk RU':>10}")


for name, emb_bge, emb_dv_idx in [
    ("caption",  emb_caption,  emb_caption_dv),
    ("ocr",      emb_ocr,      emb_ocr_dv),
    ("keywords", emb_keywords, emb_keywords_dv),
]:
    en_h, _ = calc_hits(en_emb, emb_bge, valid_queries)
    ru_h, _ = calc_hits(ru_emb, emb_bge, valid_queries)
    en_h_dv, _ = calc_hits(en_emb_dv, emb_dv_idx, valid_queries)
    ru_h_dv, _ = calc_hits(ru_emb_dv, emb_dv_idx, valid_queries)
    
    print(f"{name:<15} {en_h[5]:>10.0%} {ru_h[5]:>10.0%} {en_h_dv[5]:>10.0%} {ru_h_dv[5]:>10.0%}")





Индекс           bge-m3 EN  bge-m3 RU  deepvk EN  deepvk RU
caption                39%        37%        40%        37%
ocr                    20%        27%        18%        24%
keywords               18%        16%        24%        16%


## 9. Итоговый выбор модели

После сравнения на всех индексах — выбираем финальную модель.
Если deepvk лучше — пересохраняем эмбеддинги.


In [ ]:

SAVE_DEEPVK = False  

if SAVE_DEEPVK:
    import faiss
    OUT = Path("data/processed")
    np.save(OUT / "emb_ocr.npy", emb_ocr_dv)
    np.save(OUT / "emb_caption.npy", emb_caption_dv)
    np.save(OUT / "emb_keywords.npy", emb_keywords_dv)
    
    for name, emb in [("ocr", emb_ocr_dv), ("caption", emb_caption_dv), ("keywords", emb_keywords_dv)]:
        idx = faiss.IndexFlatIP(emb.shape[1])
        idx.add(emb)
        faiss.write_index(idx, str(OUT / f"faiss_{name}.index"))
        print(f"Пересохранён: faiss_{name}.index ({idx.ntotal} vectors)")
    
    print("Финальная модель: deepvk/USER-bge-m3")
else:
    print("Финальная модель: bge-m3 (без изменений)")


Финальная модель: bge-m3 (без изменений)
